In [1]:
from pathlib import Path
import re
import json 
import sys
import os

import matplotlib
import matplotlib.pyplot as plt
# %matplotlib notebook
%matplotlib inline
import numpy as np
import pandas as pd
import seaborn as sns  
import pickle as pkl
from tqdm import tqdm
from tensorboard.backend.event_processing import event_accumulator
from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import as_completed

module_path = os.path.abspath(os.path.join('../'))
if module_path not in sys.path:
    sys.path.append(module_path)


In [2]:
font_size = 18
matplotlib.rcParams['font.size'] = font_size
matplotlib.rcParams["font.weight"] = "bold"
matplotlib.rcParams['text.usetex'] = True
matplotlib.rcParams['axes.labelsize'] = font_size
matplotlib.rcParams['axes.titlesize'] = 28
matplotlib.rcParams['legend.fontsize'] = font_size
matplotlib.rcParams['xtick.labelsize'] = font_size
matplotlib.rcParams['ytick.labelsize'] = font_size
#matplotlib.rcParams['text.latex.preamble']=r'\usepackage{amsmath} \DeclareMathOperator{\F}{F} \DeclareMathOperator{\GMF}{GMF} \DeclareMathOperator{\MLP}{MLP}'

In [3]:
def read_tb_file(tb_file):
    ea = event_accumulator.EventAccumulator(str(tb_file),
                                            size_guidance={ 
                                                event_accumulator.COMPRESSED_HISTOGRAMS: 0,
                                                event_accumulator.IMAGES: 0,
                                                event_accumulator.AUDIO: 0,
                                                event_accumulator.SCALARS: 0,
                                                event_accumulator.HISTOGRAMS: 0,})
    ea.Reload()
    return ea

def create_df_from_ea(ea, tag):
    steps = [x.step for x in ea.Scalars(tag)]
    values = [x.value for x in ea.Scalars(tag)]
    t0 = ea.Scalars(tag)[0].wall_time
    time = [x.wall_time-t0 for x in ea.Scalars(tag)]

    df = pd.DataFrame({"Time": time, "Step": steps, "Value": values})
    return df

def get_hparams_from_ea(ea):
    hparams_string = ea.Tensors('hyperparameters/text_summary')[0][2].string_val[0].decode()
    hparams_l = hparams_string.replace('|\n|',',').replace('|',':').replace(':param:value,-:-,','').split(',')
    keys = [i.split(':')[0] for i in hparams_l if len(i.split(':'))>1]
    values = [i.split(':')[1] for i in hparams_l if len(i.split(':'))>1]
        
    d = {k:v for k,v in zip(keys,values)}
    
    return d

def create_mu_std_df_from_ea(ea_dict, tag, filter_best=False, n_best=5):
    min_len = 1e6
    
    df_list = []
    for name, ea in ea_dict.items():
        df = create_df_from_ea(ea, tag)
        if len(df) < min_len:
            min_len = len(df)
        df_list.append(df)

    return create_mu_std_df(df_list, filter_best, n_best)

def create_mu_std_df(df_list, filter_best=False, n_best=5):
    print(df_list)
    val_list = []
    for df in df_list:
        val_list.append(df["Value"])
    val_list = np.array(val_list)
    
    means = np.mean(val_list, axis=-1)
    idx = np.argsort(-means)
    if filter_best:
        vals = val_list[idx[:n_best]]
    else:
        vals = val_list
    x_axis = df_list[0]["Step"][:min_len]
    
    mu = np.mean(vals, axis=0)
    median = np.median(vals, axis=0)
    std = np.std(vals, axis=0)
    q1 = np.quantile(vals, 0.25, axis=0)
    q3 = np.quantile(vals, 0.75, axis=0)
    best = val_list[idx[0]]
    return pd.DataFrame({"Step": x_axis, "mu": mu, "std": std, "best": best, "median": median, "q1": q1, "q3": q3})

def load_dirs(tasks_dir, tags, save_path=None):
    # init dicts
    df_dict = {}
    for tag in tags:
        df_dict[tag] = {}
        
    for task_dir in tqdm(tasks_dir.iterdir()):        
        ea_dict = {}

        if len(list((task_dir).iterdir())) > 0:
            try:
                ea_dict[task_dir] = read_tb_file(task_dir)
            except:
                print(f"WARNING: No data in {run_dir}. Skipped.")
                continue

        if ea_dict:
            for tag in tags:
                df_dict[tag][task_dir.name] = create_mu_std_df(
                    ea_dict, tag, filter_best=True, n_best=5).copy()
    
    if save_path:
        with open(save_path, 'wb') as handle:
            pkl.dump(df_dict, handle, protocol=pkl.HIGHEST_PROTOCOL)
    return df_dict

def plot_mu_std(x_axis, mu, q1=None, q3=None, std=None, fig=None, title=None, label='', yuplim=None, 
                ydownlim=None, interpolate=False, max_num_tasks=None):
    if fig is None:
        fig = plt.figure(figsize=[12,10])
        
    if interpolate:
        from scipy.interpolate import make_interp_spline, BSpline
        std = std[::2]
        new_size = std.shape[0]
        
        xnew = np.linspace(x_axis.min(), x_axis.max(), new_size)
        spl = make_interp_spline(x_axis, mu, k=3)  # type: BSpline
        power_smooth = spl(xnew)
        
        x_axis = xnew
        power_smooth = np.clip(power_smooth, np.min(mu), np.max(mu))
        mu = power_smooth

    if std is not None:
        uperror = mu+std
        downerror = mu-std
        
        uperror = np.clip(uperror, np.min(mu), np.max(mu))
        downerror = np.clip(downerror, np.min(mu), np.max(mu))
    else:
        uperror = q3
        downerror = q1    
    
    if yuplim is not None:
        uperror = np.where(uperror > yuplim, yuplim, uperror)
    if ydownlim is not None:
        downerror = np.where(downerror < ydownlim, ydownlim, downerror)
    plt.plot(x_axis, mu, label=label)
    plt.fill_between(x_axis, downerror, uperror, alpha=.1)
    plt.xlabel("Timesteps", fontsize=16)
    plt.yticks(fontsize=16)
    plt.xticks(fontsize=16)
    
    if title is not None:
        plt.title(title)

#     plt.ticklabel_format(axis="x", style="sci", scilimits=(0,0))
    if max_num_tasks:
        plt.yticks(np.arange(max_num_tasks+1), 
                   ["{0:1.1f}\%".format(100.0 * i/max_num_tasks) for i in np.arange(max_num_tasks+1)])
#     plt.legend(loc="lower right")
#     plt.legend(loc="upper left")
    
    return fig

def create_mean_std_plot(ea_dict, tag,  title, smoothing=3.0, savefig_fmt=None, filter_best=False):
    df = create_mu_std_df(ea_dict, tag)
    fig = plot_mu_std(title, 'forward',df["steps"], df["mu"], df["std"])
    if savefig_fmt is not None:
        plt.savefig(title + '.' + savefig_fmt, format=savefig_fmt, dpi=300)
        
    return fig

def parse_fname(fname):    
    intgr = re.findall(r'\d+', fname)
    floats = re.findall(r"\d+\.\d+", fname)
    name = re.findall(r"[a-zA-Z0-9]*-[-a-zA-Z0-9]", fname)
    seed = intgr[-4]
    n_epc = intgr[-1]
    n_epi = intgr[-2]
    if len(floats) > 0:
        irm_coeff = floats[0]
    else:
        irm_coeff = 0
        
    return int(seed), float(irm_coeff), name

In [4]:
tags = ["charts/episodic_return", "irl/gail_bce_loss"]
run_dir_gail = Path("20230217")      
run_dir_airl = Path("202302x")      

run_dir_latest = Path("runs/210523_21")
#run_dir_swil = Path("runs/2905_swil_ae")
run_dir_swil = Path("../runs/2209_sac_24tanh/")

#df_dict = load_dirs(run_dir, tags)

In [5]:
###

In [ ]:
ea_dict = {}

def load_to_dict(p):
    print(p)
    ea = read_tb_file(p)
    ea_dict[p] = ea
    
    return ea 

with ThreadPoolExecutor(20) as executor:
    futures = [executor.submit(load_to_dict, p) for p in run_dir_swil.iterdir()]

for f in futures:
    print(f.result())

../runs/2209_sac_24tanh/20230921_205428_047240_HalfCheetah-v3__sac_swil_seed_322_irm0_lip0___exp_id__swil_24tanh___seed_322__max_q_len_10__n_demos_1__subsample_100__use_actions_False__swil_period_100.0__verbose_True../runs/2209_sac_24tanh/20230921_165702_484038_Ant-v3__sac_swil_seed_515_irm0_lip0_ua__exp_id__swil_24tanh___seed_515__env_id__Ant-v3___max_q_len_10__replace_atoms_True__n_demos_4__swil_period_100.0__verbose_True

../runs/2209_sac_24tanh/20230921_221205_320398_HalfCheetah-v3__sac_swil_seed_72_irm0_lip0_ua__exp_id__swil_24tanh___seed_72__max_q_len_5__replace_atoms_True__n_demos_1__subsample_20__swil_period_100.0__verbose_True
../runs/2209_sac_24tanh/20230921_210311_184062_Hopper-v3__sac_swil_seed_322_irm0_lip0_ua__exp_id__swil_24tanh___seed_322__env_id__Hopper-v3___max_q_len_10__n_demos_1__subsample_100__use_next_obs_True__swil_period_100.0__verbose_True
../runs/2209_sac_24tanh/20230922_201858_941247_HalfCheetah-v3__sac_gail_seed_234_irm0_lip0_ua__exp_id__swil_24tanh_2___seed

In [ ]:
print(len(ea_dict.keys()))

In [ ]:
seeds = [515, 234, 894, 322, 72]
# seeds = [234]
# algos = ['ppo', 'sac']
# algos = ['ppo', 'sac']
algos = ['sac']
disc_types = ['gail']
# irm_coeffs = [0.0, 0.1, 1.0, 10.0, 100.0]
# n_projs = [1,5,20,50]
# n_projs = [1,20,100]
# n_projs = [1]
# irm_coeffs = [0.1, 1.0, 10.0, 50.0, 100.0]
# irm_coeffs = [0.0, 1.0, 100.0]
# irm_coeffs = [50.0, 100.0]
envs = ['Ant-v3', 'Walker2d-v3', 'Hopper-v3', 'HalfCheetah-v3', 'Humanoid-v3']
# envs = ['Humanoid-v3']
# lins = ['--use_spectral_norm', '--use_weight_norm', '']
# lins = ['--use_spectral_norm', '']
# lins = ['--use_spectral_norm', '']
lins = ['', '--irm_coeff=1.0', '--lip_coeff=1.0', '--use_spectral_norm', '--use_absorb_wrapper']
# lins = ['--use_weight_norm --linear_proj']
# usenobs = ['--use_next_obs', '', '--use_next_obs --use_dones']
# usenobs = ['--use_next_obs', '', '--use_next_obs --use_dones']
usenobs = ['', '--use_actions']
# envs = ['Walker2d-v3', 'Hopper-v3', 'HalfCheetah-v3']
# envs = ['HalfCheetah-v3']
# use_acs = ['', '--use_actions']
# use_acs = ['--use_actions', '']
# use_acs = ['--use_actions']
use_acs = ['']
# n_demos = [1,4,10]
# n_demos = [1,4,10,25]
subsamplings = [1,20,100]
periods = [100, -1]
n_demos = [1,4,10]
# n_envs = [1,8,16]
# ent_coeffs = [0.01]
ent_coeffs = [0.0]
# disc_nonlins = ['tanh', 'prelu', 'relu']
# disc_nonlins = ['tanh', 'leakyrelu']
disc_nonlins = ['tanh']
# lip_coeffs = [1.0]
# lip_coeffs = [0.0, 1.0]
# lip_coeff = ['--use_actions']
qlens = [1]

d = {}

for env_name in envs:
    d[env_name] = {}
    for seed in seeds:
        d[env_name][seed] = {}
        for ss in subsamplings:
            d[env_name][seed][f"ss_{ss}"] = {}
            for n_demo in n_demos:
                d[env_name][seed][f"ss_{ss}"][f"nd_{n_demo}"]  = {}
                for lin in lins:
                    d[env_name][seed][f"ss_{ss}"][f"nd_{n_demo}"][lin]  = {}
                    for ua in usenobs:
                        d[env_name][seed][f"ss_{ss}"][f"nd_{n_demo}"][lin][ua]  = {}
                        for d_nl in disc_nonlins:                        
                            d[env_name][seed][f"ss_{ss}"][f"nd_{n_demo}"][lin][ua][d_nl]  = {}
                            for disc_type in disc_types:
                                d[env_name][seed][f"ss_{ss}"][f"nd_{n_demo}"][lin][ua][d_nl][disc_type]  = {}
                                for per in periods:
                                    d[env_name][seed][f"ss_{ss}"][f"nd_{n_demo}"][lin][ua][d_nl][disc_type][f"per_{per}"]  = {}
                                    for qlen in qlens:
                                        d[env_name][seed][f"ss_{ss}"][f"nd_{n_demo}"][lin][ua][d_nl][disc_type][f"per_{per}"][f"ql_{qlen}"]  = {}

                            
                        
def parse_fname_irm(fname):    
    intgr = re.findall(r'\d+', fname)
    floats = re.findall(r"\d+\.\d+", fname)
    name = re.findall(r"[a-zA-Z0-9]*-[-a-zA-Z0-9]", fname)

    m = re.search('seed_(\d+)', fname, re.IGNORECASE)
    seed = int(m.group(1))
    #m = re.search('irm(\d+)', fname, re.IGNORECASE)
    #print(m)
    #irm = int(m.group(1))

    if len(floats) > 0:
        irm_coeff = floats[0]
    else:
        irm_coeff = 0

    return int(seed), float(irm_coeff), name

def parse_fname_il(fname):    
    intgr = re.findall(r'\d+', fname)
    floats = re.findall(r"\d+\.\d+", fname)
    name = re.findall(r"[a-zA-Z0-9]*-[-a-zA-Z0-9]", fname)

    m = re.search('seed_(\d+)', fname, re.IGNORECASE)
    seed = int(m.group(1))
    np = re.search('np(\d+)', fname, re.IGNORECASE)
    #np = int(np.group(1))
    np = 0
    #m = re.search('irm(\d+)', fname, re.IGNORECASE)
    #irm = int(m.group(1))

    return int(seed), np, name

def parse_fname_baselines(fname):    
    intgr = re.findall(r'\d+', fname)
    floats = re.findall(r"\d+\.\d+", fname)
    name = re.findall(r"[a-zA-Z0-9]*-[-a-zA-Z0-9]", fname)

    m = re.search('seed_(\d+)', fname, re.IGNORECASE)
    seed = int(m.group(1))
    dt = ''
    if 'airl' in fname:
        dt = 'airl'
    elif 'gail' in fname:
        dt = 'gail'
    else:
        dt = 'pwil'
        
    #m = re.search('irm(\d+)', fname, re.IGNORECASE)
    #irm = int(m.group(1))

    return int(seed), dt, name

In [ ]:
d_gail = d.copy()

In [ ]:
seeds = [515, 234, 894, 322, 72]
# seeds = [234]
# algos = ['ppo', 'sac']
# algos = ['ppo', 'sac']
algos = ['sac']
disc_types = ['swil']
# irm_coeffs = [0.0, 0.1, 1.0, 10.0, 100.0]
# n_projs = [1,5,20,50]
# n_projs = [1,20,100]
# n_projs = [1]
# irm_coeffs = [0.1, 1.0, 10.0, 50.0, 100.0]
# irm_coeffs = [0.0, 1.0, 100.0]
# irm_coeffs = [50.0, 100.0]
envs = ['Ant-v3', 'Walker2d-v3', 'Hopper-v3', 'HalfCheetah-v3', 'Humanoid-v3']
# envs = ['Humanoid-v3']
# lins = ['--use_spectral_norm', '--use_weight_norm', '']
# lins = ['--use_spectral_norm', '']
# lins = ['--use_spectral_norm', '']
lins = ['', '--replace_atoms']
# lins = ['']
# lins = ['--use_weight_norm --linear_proj']
# usenobs = ['--use_next_obs', '', '--use_next_obs --use_dones']
# usenobs = ['--use_next_obs', '', '--use_next_obs --use_dones']
usenobs = ['', '--use_actions', '--use_next_obs--use_actions', '--use_next_obs--use_actions--use_dones']
# envs = ['Walker2d-v3', 'Hopper-v3', 'HalfCheetah-v3']
# envs = ['HalfCheetah-v3']
# use_acs = ['', '--use_actions']
# use_acs = ['--use_actions', '']
# use_acs = ['--use_actions']
use_acs = ['']
# n_demos = [1,4,10]
# n_demos = [1,4,10,25]
n_demos = [1,4,10]
# n_envs = [1,8,16]
# ent_coeffs = [0.01]
qlens = [1,5,10]
subsamplings = [1,20,100]
ent_coeffs = [0.0]
periods = [100]
disc_nonlins = ['tanh']
# disc_nonlins = ['tanh', 'prelu', 'relu']
# disc_nonlins = ['tanh', 'leakyrelu']
# lip_coeffs = [1.0]
# lip_coeffs = [0.0, 1.0]
# lip_coeff = ['--use_actions']

d = {}

for env_name in envs:
    d[env_name] = {}
    for seed in seeds:
        d[env_name][seed] = {}
        for ss in subsamplings:
            d[env_name][seed][f"ss_{ss}"] = {}
            for n_demo in n_demos:
                d[env_name][seed][f"ss_{ss}"][f"nd_{n_demo}"]  = {}
                for lin in lins:
                    d[env_name][seed][f"ss_{ss}"][f"nd_{n_demo}"][lin]  = {}
                    for ua in usenobs:
                        d[env_name][seed][f"ss_{ss}"][f"nd_{n_demo}"][lin][ua]  = {}
                        for d_nl in disc_nonlins:                        
                            d[env_name][seed][f"ss_{ss}"][f"nd_{n_demo}"][lin][ua][d_nl]  = {}
                            for disc_type in disc_types:
                                d[env_name][seed][f"ss_{ss}"][f"nd_{n_demo}"][lin][ua][d_nl][disc_type]  = {}
                                for per in periods:
                                    d[env_name][seed][f"ss_{ss}"][f"nd_{n_demo}"][lin][ua][d_nl][disc_type][f"per_{per}"]  = {}
                                    for qlen in qlens:
                                        d[env_name][seed][f"ss_{ss}"][f"nd_{n_demo}"][lin][ua][d_nl][disc_type][f"per_{per}"][f"ql_{qlen}"]  = {}

                        
def parse_fname_irm(fname):    
    intgr = re.findall(r'\d+', fname)
    floats = re.findall(r"\d+\.\d+", fname)
    name = re.findall(r"[a-zA-Z0-9]*-[-a-zA-Z0-9]", fname)

    m = re.search('seed_(\d+)', fname, re.IGNORECASE)
    seed = int(m.group(1))
    #m = re.search('irm(\d+)', fname, re.IGNORECASE)
    #print(m)
    #irm = int(m.group(1))

    if len(floats) > 0:
        irm_coeff = floats[0]
    else:
        irm_coeff = 0

    return int(seed), float(irm_coeff), name

def parse_fname_il(fname):    
    intgr = re.findall(r'\d+', fname)
    floats = re.findall(r"\d+\.\d+", fname)
    name = re.findall(r"[a-zA-Z0-9]*-[-a-zA-Z0-9]", fname)

    m = re.search('seed_(\d+)', fname, re.IGNORECASE)
    seed = int(m.group(1))
    np = re.search('np(\d+)', fname, re.IGNORECASE)
    #np = int(np.group(1))
    np = 0
    #m = re.search('irm(\d+)', fname, re.IGNORECASE)
    #irm = int(m.group(1))

    return int(seed), np, name

def parse_fname_baselines(fname):    
    intgr = re.findall(r'\d+', fname)
    floats = re.findall(r"\d+\.\d+", fname)
    name = re.findall(r"[a-zA-Z0-9]*-[-a-zA-Z0-9]", fname)

    m = re.search('seed_(\d+)', fname, re.IGNORECASE)
    seed = int(m.group(1))
    dt = ''
    if 'airl' in fname:
        dt = 'airl'
    elif 'gail' in fname:
        dt = 'gail'
    else:
        dt = 'pwil'
        
    #m = re.search('irm(\d+)', fname, re.IGNORECASE)
    #irm = int(m.group(1))

    return int(seed), dt, name

In [ ]:
d_swil = d.copy()

In [ ]:
for k, v in ea_dict.items():
    if 'sac_swil' in str(k):
        hparams = get_hparams_from_ea(v)
        lin_string = ""
        if hparams['replace_atoms'] == 'True':
            lin_string += "--replace_atoms"
            
        nob_string = ""
        if hparams['use_next_obs'] == 'True':
            nob_string += "--use_next_obs"
        if hparams['use_actions'] == 'True':
            nob_string += "--use_actions"
        if hparams['use_dones'] == 'True':
            nob_string += "--use_dones"

        seed, _, name = parse_fname_irm(str(k))
        
        n_demos = f"nd_{hparams['n_demos']}"
        env_name = name[0]+'3'

        if 'gail' in str(k):
            disc_type = 'gail'
        elif 'airl' in str(k):
            disc_type = 'airl'
        elif 'swil' in str(k):
            disc_type = 'swil'
        else:
            disc_type = 'wail'
        
        d_nl = hparams['disc_nonlin']
        ss = int(hparams['subsample'])
        qlen = int(hparams['max_q_len'])

        #print(env_name, seed, n_proj, n_demos, lin_string, nob_string, d_nl)
        d_swil[env_name][seed][f"ss_{ss}"][n_demos][lin_string][nob_string][d_nl][disc_type][f"per_{per}"][f"ql_{qlen}"]  = v


In [ ]:
for k, v in ea_dict.items():
    if 'gail' in str(k):
        hparams = get_hparams_from_ea(v)
        lin_string = ""
        if hparams['use_weight_norm'] == 'True':
            lin_string += "--use_weight_norm"
        if hparams['use_spectral_norm'] == 'True':
            lin_string += "--use_spectral_norm"
        if hparams['use_absorb_wrapper'] == 'True':
            lin_string += "--use_absorb_wrapper"
        if float(hparams['lip_coeff']) > 0:            
            lin_string += f"--lip_coeff={hparams['lip_coeff']}"
        if float(hparams['irm_coeff']) > 0:            
            lin_string += f"--irm_coeff={hparams['irm_coeff']}"

        nob_string = ""
        #if hparams['use_next_obs'] == 'True':
        #    nob_string += "--use_next_obs"
        if hparams['use_actions'] == 'True':
            nob_string += "--use_actions"
        #if hparams['use_dones'] == 'True':
        #    nob_string += "--use_dones"

        seed, _, name = parse_fname_irm(str(k))        
        ec = hparams['ent_coef']
        n_demos = f"nd_{hparams['n_demos']}"
        env_name = name[0]+'3'

        if 'gail' in str(k):
            disc_type = 'gail'
        elif 'airl' in str(k):
            disc_type = 'airl'
        elif 'swil' in str(k):
            disc_type = 'swil'
        else:
            disc_type = 'wail'
        
        d_nl = hparams['disc_nonlin']
        ss = int(hparams['subsample'])
        per = int(hparams['disc_update_period'])
        #print(env_name, seed, n_proj, n_demos, lin_string, nob_string, d_nl)
        #print(k)
        #print(env_name, seed, ss, n_demos, lin_string, nob_string, d_nl, per)
        d_gail[env_name][seed][f"ss_{ss}"][n_demos][lin_string][nob_string][d_nl][disc_type][f"per_{per}"][f"ql_1"]  = v


In [ ]:
df_dict_ciil = d
tags = ["charts/episodic_return", "irl/gail_bce_loss", "eval/mu"]

#nonlin="leakyrelu"
#lin = '--linear_proj'
#nob = ''

X = ['gail']
subsamplings = [1,20,100]
#periods = [100, -1]
periods = [-1]
#lins = ['', '--irm_coeff=1.0', '--lip_coeff=1.0', '--use_spectral_norm', '--use_absorb_wrapper']
lins = ['', '--irm_coeff=1.0', '--lip_coeff=1.0', '--use_absorb_wrapper']
usenobs = ['', '--use_actions']

print(lins)

for algo in X:
    for ss in subsamplings:
        for nob in usenobs:
            for per in periods:
                for nonlin in disc_nonlins:
                    fig, axs = plt.subplots(1,5, figsize=(24,4))
                    fig.subplots_adjust(top=0.88)
                    fig.suptitle(f"{algo}_subsmpl:{ss}_{nob}_nonlin:{nonlin}_period:{per}")
                    plt.style.use('ggplot')
                    #print(lin, nob, nonlin)
                    #plt.title(lin + nob + nonlin)
                    for i, env in enumerate(envs):

                        X_axis = np.arange(len(lins))

                        axs[i].set_title(env)
                        X_index = []
                        Y_mean = []
                        Y_std = []
                        ndem = [1,4,10]

                        for j,n_d in enumerate(ndem):
                            Y = []
                            Y_ = []

                            for k,lin in enumerate(lins):
                                val_list = []
                                for seed in [515, 234, 894, 322, 72]:
                                    ea = d_gail[env][seed][f"ss_{ss}"][f"nd_{n_d}"][lin][nob][nonlin][algo][f"per_{per}"][f"ql_1"]
                                    if not isinstance(ea,dict):
                                        df = create_df_from_ea(ea, tags[-1])
                                        val_list.append(np.max(df['Value']))
                                
                                if len(val_list) > 0:
                                    Y.append(np.mean(val_list))
                                    Y_.append(np.std(val_list))
                                else:
                                    Y.append(0)            
                                    Y_.append(0)

                            #print(Y, Y_)

                            axs[i].bar(X_axis-0.2+j*0.2, Y, 0.2, label=f"{n_d} traj") 
                            axs[i].errorbar(X_axis-0.2+j*0.2, Y, yerr=Y_, fmt=".", color='k') 


                    axs[0].set_ylabel("Cumulative reward")
                    axs[-1].legend(loc='center left', bbox_to_anchor=(1, 0.5))

                    #plt.legend()
                    plt.tight_layout()

In [ ]:
tags = ["charts/episodic_return", "irl/gail_bce_loss", "eval/mu"]

#nonlin="leakyrelu"
#lin = '--linear_proj'
#nob = ''

algos = ['sac']
disc_types = ['swil']
lins = ['', '--replace_atoms']
usenobs = ['', '--use_actions', '--use_next_obs--use_actions', '--use_next_obs--use_actions--use_dones']

qlens = [1,5,10]
subsamplings = [1,20,100]
periods = [100]
disc_nonlins = ['tanh']
print(lins)

for algo in X:
    for ss in subsamplings:
        for nob in usenobs:
            for per in periods:
                for ql in qlens:
                    for nonlin in disc_nonlins:
                        fig, axs = plt.subplots(1,5, figsize=(24,4))
                        fig.subplots_adjust(top=0.88)
                        fig.suptitle(f"{algo}_subsmpl:{ss}_{nob}_nonlin:{nonlin}_period:{per}_qlen:{ql}")
                        plt.style.use('ggplot')
                        #print(lin, nob, nonlin)
                        #plt.title(lin + nob + nonlin)
                        for i, env in enumerate(envs):

                            X_axis = np.arange(len(lins))

                            axs[i].set_title(env)
                            X_index = []
                            Y_mean = []
                            Y_std = []
                            ndem = [1,4,10]

                            for j,n_d in enumerate(ndem):
                                Y = []
                                Y_ = []

                                for k,lin in enumerate(lins):
                                    val_list = []
                                    for seed in [515, 234, 894, 322, 72]:
                                        ea = d_swil[env][seed][f"ss_{ss}"][f"nd_{n_d}"][lin][nob][nonlin]['swil'][f"per_{per}"][f"ql_{ql}"]
                                        if not isinstance(ea,dict):
                                            df = create_df_from_ea(ea, tags[-1])
                                            val_list.append(np.max(df['Value']))

                                    if len(val_list) > 0:
                                        Y.append(np.mean(val_list))
                                        Y_.append(np.std(val_list))
                                    else:
                                        Y.append(0)            
                                        Y_.append(0)

                                #print(f"{env_name}[seed][f"ss_{ss}"][f"nd_{n_d}"][lin][nob][nonlin]['swil'][f"per_{per}"][f"ql_{ql}"Y, Y_)

                                axs[i].bar(X_axis-0.2+j*0.2, Y, 0.2, label=f"{n_d} traj") 
                                axs[i].errorbar(X_axis-0.2+j*0.2, Y, yerr=Y_, fmt=".", color='k') 


                        axs[0].set_ylabel("Cumulative reward")
                        axs[-1].legend(loc='center left', bbox_to_anchor=(1, 0.5))

                        #plt.legend()
                        plt.tight_layout()

In [ ]:
### cum.rew. plots

### GAIL

In [ ]:
tags = ["charts/episodic_return", "irl/gail_bce_loss", "eval/mu"]

#nonlin="leakyrelu"
#lin = '--linear_proj'
#nob = ''

algos = ['sac']
disc_types = ['gail']
lins = ['', '--irm_coeff=1.0', '--lip_coeff=1.0', '--use_spectral_norm', '--use_absorb_wrapper']
usenobs = ['', '--use_actions']

qlens = [1]
subsamplings = [1,20,100]
periods = [-1]
disc_nonlins = ['tanh']
print(lins)
n_demos = [1,4,10]

d_mustd_gail = {}
for env in envs:
    d_mustd_gail[env] = {}
    for ss in subsamplings:
        d_mustd_gail[env][f"ss_{ss}"] = {}
        for n_demo in n_demos:
            d_mustd_gail[env][f"ss_{ss}"][f"nd_{n_demo}"]  = {}
            for lin in lins:
                d_mustd_gail[env][f"ss_{ss}"][f"nd_{n_demo}"][lin]  = {}
                for ua in usenobs:
                    d_mustd_gail[env][f"ss_{ss}"][f"nd_{n_demo}"][lin][ua]  = {}
                    for per in periods:
                        d_mustd_gail[env][f"ss_{ss}"][f"nd_{n_demo}"][lin][ua][f"per_{per}"]  = {}
                        for qlen in qlens:
                            d_mustd_gail[env][f"ss_{ss}"][f"nd_{n_demo}"][lin][ua][f"per_{per}"][f"ql_{qlen}"]  = {}

subsample_plot = 2

for algo in X:
    for ss in subsamplings:
        for nob in usenobs:
            for per in periods:
                for ql in qlens:
                    for nonlin in disc_nonlins:
                        fig, axs = plt.subplots(1,5, figsize=(24,4))
                        fig.subplots_adjust(top=0.88)
                        fig.suptitle(f"{algo}_subsmpl:{ss}_{nob}_nonlin:{nonlin}_period:{per}_qlen:{ql}")
                        plt.style.use('ggplot')
                        #print(lin, nob, nonlin)
                        #plt.title(lin + nob + nonlin)
                        for i, env in enumerate(envs):

                            X_axis = np.arange(len(lins))

                            axs[i].set_title(env)
                            X_index = []
                            Y_mean = []
                            Y_std = []
                            ndem = [1,4,10]
                            #ndem = [10]

                            for j,n_d in enumerate(ndem):
                                Y = []
                                Y_ = []

                                for k,lin in enumerate(lins):
                                    val_list = []
                                    min_len = 1000
                                    df_list = []
                                    for seed in [515, 234, 894, 322, 72]:
                                        ea = d_gail[env][seed][f"ss_{ss}"][f"nd_{n_d}"][lin][nob][nonlin]['gail'][f"per_{per}"][f"ql_{ql}"]
                                        if not isinstance(ea,dict): # prevents case if we have empty dict
                                            df = create_df_from_ea(ea, tags[-1])
                                            df_list.append(df)
                                            min_len = min(len(df['Value']), min_len)

                                    for df_ in df_list:
                                        val_list.append(df_['Value'][:min_len])

                                    if len(val_list) > 0:
                                        vals = np.array(val_list)
                                        mu = np.mean(vals,0)[::subsample_plot]
                                        std = np.std(vals,0)[::subsample_plot]
                                        d_mustd_gail[env][f"ss_{ss}"][f"nd_{n_d}"][lin][nob][f"per_{per}"][f"ql_{qlen}"] = (mu,std)
                                        if lin == '':
                                            lin_label = 'no_atom_repl'
                                        else:
                                            lin_label = lin
                                    
                                        axs[i].plot(mu, label=f"nd_{n_d}_{lin_label}")
                                        axs[i].fill_between(np.arange(len(mu)),mu+std, mu-std, alpha=0.2)

                                    else:
                                        Y.append(0)            
                                        Y_.append(0)

                                    #print(Y, Y_)

                                #axs[i].bar(X_axis-0.2+j*0.2, Y, 0.2) 
                                #axs[i].errorbar(X_axis-0.2+j*0.2, Y, yerr=Y_, fmt=".", color='k') 

                            axs[0].set_ylabel("Cumulative reward")
                            axs[-1].legend(loc='center left', bbox_to_anchor=(1, 0.5))

                            #plt.legend()
                            plt.tight_layout()

## SWIL

In [ ]:
tags = ["charts/episodic_return", "irl/gail_bce_loss", "eval/mu"]

#nonlin="leakyrelu"
#lin = '--linear_proj'
#nob = ''

algos = ['sac']
disc_types = ['swil']
lins = ['', '--replace_atoms']
usenobs = ['', '--use_actions', '--use_next_obs--use_actions', '--use_next_obs--use_actions--use_dones']

qlens = [1,5,10]
subsamplings = [1,20,100]
periods = [100]
disc_nonlins = ['tanh']
print(lins)

d_mustd_swil = {}
for env in envs:
    d_mustd_swil[env] = {}
    for ss in subsamplings:
        d_mustd_swil[env][f"ss_{ss}"] = {}
        for n_demo in n_demos:
            d_mustd_swil[env][f"ss_{ss}"][f"nd_{n_demo}"]  = {}
            for lin in lins:
                d_mustd_swil[env][f"ss_{ss}"][f"nd_{n_demo}"][lin]  = {}
                for ua in usenobs:
                    d_mustd_swil[env][f"ss_{ss}"][f"nd_{n_demo}"][lin][ua]  = {}
                    for per in periods:
                        d_mustd_swil[env][f"ss_{ss}"][f"nd_{n_demo}"][lin][ua][f"per_{per}"]  = {}
                        for qlen in qlens:
                            d_mustd_swil[env][f"ss_{ss}"][f"nd_{n_demo}"][lin][ua][f"per_{per}"][f"ql_{qlen}"]  = {}

subsample_plot = 2

for algo in disc_types:
    for ss in subsamplings:
        for nob in usenobs:
            for per in periods:
                for ql in qlens:
                    for nonlin in disc_nonlins:
                        fig, axs = plt.subplots(1,5, figsize=(24,4))
                        fig.subplots_adjust(top=0.88)
                        fig.suptitle(f"{algo}_subsmpl:{ss}_{nob}_nonlin:{nonlin}_period:{per}_qlen:{ql}")
                        plt.style.use('ggplot')
                        #print(lin, nob, nonlin)
                        #plt.title(lin + nob + nonlin)
                        for i, env in enumerate(envs):

                            X_axis = np.arange(len(lins))

                            axs[i].set_title(env)
                            X_index = []
                            Y_mean = []
                            Y_std = []
                            ndem = [1,4,10]

                            for j,n_d in enumerate(ndem):
                                Y = []
                                Y_ = []

                                for k,lin in enumerate(lins):
                                    val_list = []
                                    min_len = 1000
                                    df_list = []
                                    for seed in [515, 234, 894, 322, 72]:
                                        ea = d_swil[env][seed][f"ss_{ss}"][f"nd_{n_d}"][lin][nob][nonlin]['swil'][f"per_{per}"][f"ql_{ql}"]
                                        if not isinstance(ea,dict): # prevents case if we have empty dict
                                            df = create_df_from_ea(ea, tags[-1])
                                            df_list.append(df)
                                            min_len = min(len(df['Value']), min_len)

                                    for df_ in df_list:
                                        val_list.append(df_['Value'][:min_len])

                                    if len(val_list) > 0:
                                        vals = np.array(val_list)
                                        mu = np.mean(vals,0)[::subsample_plot]
                                        std = np.std(vals,0)[::subsample_plot]
                                        d_mustd_swil[env][f"ss_{ss}"][f"nd_{n_d}"][lin][nob][f"per_{per}"][f"ql_{ql}"] = (mu,std)
                                        if lin == '':
                                            lin_label = 'no_atom_repl'
                                        else:
                                            lin_label = lin
                                    
                                        axs[i].plot(mu, label=f"nd_{n_d}_{lin_label}")
                                        axs[i].fill_between(np.arange(len(mu)),mu+std, mu-std, alpha=0.2)

                                    else:
                                        Y.append(0)            
                                        Y_.append(0)

                                    #print(Y, Y_)

                                #axs[i].bar(X_axis-0.2+j*0.2, Y, 0.2) 
                                #axs[i].errorbar(X_axis-0.2+j*0.2, Y, yerr=Y_, fmt=".", color='k') 

                            axs[0].set_ylabel("Cumulative reward")
                            axs[-1].legend(loc='center left', bbox_to_anchor=(1, 0.5))

                            #plt.legend()
                            plt.tight_layout()

## Plot to finally combine it all...

In [ ]:
def plot_comp(d_ms_swil, d_ms_gail, n_demo, swil_ql, ss):
    envs = ['Ant-v3', 'Walker2d-v3', 'Hopper-v3', 'HalfCheetah-v3', 'Humanoid-v3']
    fig, axs = plt.subplots(1,5, figsize=(24,4))
    fig.suptitle(f"ss:{ss}_ql:{swil_ql}")
    for i, env in enumerate(envs):
        axs[i].set_title(f"{env}")

        for lin_gail in ['', '--irm_coeff=1.0', '--lip_coeff=1.0', '--use_spectral_norm', '--use_absorb_wrapper']:
        for lin_gail in ['', '--lip_coeff=1.0', '--use_spectral_norm', '--use_absorb_wrapper']:
            mu, std = d_ms_gail[env][f"ss_{ss}"][f"nd_{n_demo}"][lin_gail][''][f"per_-1"][f"ql_1"]
            axs[i].plot(mu, label=f"gail_{lin_gail}")
            axs[i].fill_between(np.arange(len(mu)),mu+std, mu-std, alpha=0.1)
            mu, std = d_ms_gail[env][f"ss_{ss}"][f"nd_{n_demo}"][lin_gail]['--use_actions'][f"per_-1"][f"ql_1"]
            axs[i].plot(mu, label=f"gail_{lin_gail}_ua")
            axs[i].fill_between(np.arange(len(mu)),mu+std, mu-std, alpha=0.1)
        #for nob_swil in ['', '--use_actions', '--use_next_obs--use_actions', '--use_next_obs--use_actions--use_dones']:        
        for nob_swil in ['', '--use_next_obs--use_actions--use_dones']:        
            mu, std = d_ms_swil[env][f"ss_{ss}"][f"nd_{n_demo}"][''][nob_swil][f"per_100"][f"ql_{swil_ql}"]
            axs[i].plot(mu, label=f"swil_{nob_swil}")
            axs[i].fill_between(np.arange(len(mu)),mu+std, mu-std, alpha=0.1)
            mu, std = d_ms_swil[env][f"ss_{ss}"][f"nd_{n_demo}"]['--replace_atoms'][nob_swil][f"per_100"][f"ql_{swil_ql}"]
            axs[i].plot(mu, label=f"swil_{nob_swil}_ra")
            axs[i].fill_between(np.arange(len(mu)),mu+std, mu-std, alpha=0.1)    
            
    axs[0].set_ylabel("Cumulative reward")
    #axs[-1].legend(loc='center left', bbox_to_anchor=(1, 0.5))
    fig.subplots_adjust(bottom=0.3, wspace=0.33)

    axs[2].legend(loc='upper center', bbox_to_anchor=(0.5, -0.2),fancybox=False, shadow=False, ncol=3)
    #plt.legend(loc="lower left", ncol=len(df.columns))
    plt.tight_layout()
    

In [ ]:
plot_comp(d_mustd_swil, d_mustd_gail, n_demo=1, swil_ql=1, ss=1)

In [ ]:
df_dict_ciil = d
tags = ["charts/episodic_return", "irl/gail_bce_loss", "eval/mu"]

#nonlin="leakyrelu"
#lin = '--linear_proj'
#nob = ''

X = ['swil']
ec = [0.0]
subsample = 10

for algo in X:
    for ec_ in ec:
        for nob in usenobs:
            for nd in [1,4,10]:
                for nonlin in disc_nonlins:
                    fig, axs = plt.subplots(1,5, figsize=(24,4))
                    fig.subplots_adjust(top=0.88)
                    fig.suptitle(f"{algo}_ndemos_{nd}_{nonlin}")
                    plt.style.use('ggplot')
                    #print(lin, nob, nonlin)
                    #plt.title(lin + nob + nonlin)

                    for i, env in enumerate(envs):

                        X_axis = np.arange(len(lins))
                        axs[i].set_title(env)
                        Y = []
                        Y_ = []

                        for j,lin in enumerate(lins):
                            val_list = []
                            min_len = 1000
                            df_list = []
                            for seed in [515, 234, 894, 322, 72]:
                                ea = df_dict_ciil[env][seed][f"ec_{ec_}"][f"nd_{nd}"][lin][nob][nonlin][algo]

                                if not isinstance(ea,dict): # prevents case if we have empty dict
                                    df = create_df_from_ea(ea, tags[-1])
                                    df_list.append(df)
                                    min_len = min(len(df['Value']), min_len)

                            for df_ in df_list:
                                val_list.append(df_['Value'][:min_len])

                            if len(val_list) > 0:
                                vals = np.array(val_list)
                                mu = np.mean(vals,0)[::subsample]
                                std = np.std(vals,0)[::subsample]
                                if lin == '':
                                    lin_label = 'period=1'
                                else:
                                    lin_label = lin
                                axs[i].plot(mu, label=lin_label)
                                axs[i].fill_between(np.arange(len(mu)),mu+std, mu-std, alpha=0.2)

                            else:
                                Y.append(0)            
                                Y_.append(0)

                            #print(Y, Y_)

                        #axs[i].bar(X_axis-0.2+j*0.2, Y, 0.2) 
                        #axs[i].errorbar(X_axis-0.2+j*0.2, Y, yerr=Y_, fmt=".", color='k') 

                    axs[0].set_ylabel("Cumulative reward")
                    axs[-1].legend(loc='center left', bbox_to_anchor=(1, 0.5))

                    #plt.legend()
                    plt.tight_layout()

In [ ]:
def init_results_dict():
    def init_ndemos_dict():
        return {1:{'tanh': {}, 'relu': {}, 'prelu': {}}, 4: {'tanh': {}, 'relu': {}, 'prelu': {}}, 10: {'tanh': {}, 'relu': {}, 'prelu': {}}, 25: {'tanh': {}, 'relu': {}, 'prelu': {}}}
    d = {'Ant-v3': {0.0: init_ndemos_dict(), 0.01:init_ndemos_dict(), 0.1: init_ndemos_dict(), 1.0: init_ndemos_dict(), 10.0: init_ndemos_dict(), 50.0: init_ndemos_dict(), 100.0:init_ndemos_dict()},
            'HalfCheetah-v3': {0.0: init_ndemos_dict(), 0.01:init_ndemos_dict(), 0.1: init_ndemos_dict(), 1.0: init_ndemos_dict(), 10.0: init_ndemos_dict(), 50.0: init_ndemos_dict(), 100.0:init_ndemos_dict()},
                  'Hopper-v3': {0.0: init_ndemos_dict(), 0.01:init_ndemos_dict(), 0.1: init_ndemos_dict(), 1.0: init_ndemos_dict(), 10.0: init_ndemos_dict(), 50.0: init_ndemos_dict(), 100.0:init_ndemos_dict()},
                  'Walker2d-v3': {0.0: init_ndemos_dict(), 0.01:init_ndemos_dict(), 0.1: init_ndemos_dict(), 1.0: init_ndemos_dict(), 10.0: init_ndemos_dict(), 50.0: init_ndemos_dict(), 100.0:init_ndemos_dict()}}
    
    return d

def init_results_dict_small():
    d = {'Ant-v3': {0.0:{},0.01:{},0.1:{},1.0:{},10.0:{},50.0:{},100.0:{}},
                  'HalfCheetah-v3': {0.0:{},0.01:{},0.1:{},1.0:{},10.0:{},50.0:{},100.0:{}},
                  'Hopper-v3': {0.0:{},0.01:{},0.1:{},1.0:{},10.0:{},50.0:{},100.0:{}},
                  'Walker2d-v3': {0.0:{},0.01:{},0.1:{},1.0:{},10.0:{},50.0:{},100.0:{}}}
    
    return d

def init_results_dict_swil():
    def init_ndemos_dict():
        return {1:{'tanh': {}, 'relu': {}, 'prelu': {}}, 4: {'tanh': {}, 'relu': {}, 'prelu': {}}, 10: {'tanh': {}, 'relu': {}, 'prelu': {}}, 25: {'tanh': {}, 'relu': {}, 'prelu': {}}}
    d = {'Ant-v3': {1:init_ndemos_dict(),5:init_ndemos_dict(),20:init_ndemos_dict(),50:init_ndemos_dict()},
                  'HalfCheetah-v3': {1:init_ndemos_dict(),5:init_ndemos_dict(),20:init_ndemos_dict(),50:init_ndemos_dict()},
                  'Hopper-v3':{1:init_ndemos_dict(),5:init_ndemos_dict(),20:init_ndemos_dict(),50:init_ndemos_dict()},
                  'Walker2d-v3': {1:init_ndemos_dict(),5:init_ndemos_dict(),20:init_ndemos_dict(),50:init_ndemos_dict()}}
    
    return d


In [ ]:
def get_dicts_from_ea_swil(ea_dict):
    def parse_fname_swil(fname):    
        intgr = re.findall(r'\d+', fname)
        floats = re.findall(r"\d+\.\d+", fname)
        name = re.findall(r"[a-zA-Z0-9]*-[-a-zA-Z0-9]", fname)
        
        m = re.search('seed_(\d+)', fname, re.IGNORECASE)
        seed = int(m.group(1))
        m = re.search('np(\d+)', fname, re.IGNORECASE)
        np = int(m.group(1))

        offset = 0
        if "Walker2d" in fname:
            offset = 1
        #seed = intgr[5+offset]
        n_epc = intgr[-1]
        n_epi = intgr[-2]
        #np = intgr[6+offset]

        #print(fname)
        #print(name)
        #print(seed)
        #print(intgr)

        if len(floats) > 0:
            irm_coeff = floats[0]
        else:
            irm_coeff = 0

        return int(seed), int(np), name
    
    df_dict = {}

    df_dict['sac'] = init_results_dict_swil()
    df_dict['ppo'] = init_results_dict_swil()

    df_dict_flat = {}
    df_dict_max = {}
    irm_coeffs = [0.0, 0.01, 0.1, 1.0, 10.0, 50.0, 100.0]

    df_dict_max['sac'] = init_results_dict_swil()
    df_dict_max['ppo'] = init_results_dict_swil()

    hp_dict = {}
    hp_dict['sac'] = init_results_dict_swil()
    hp_dict['ppo'] = init_results_dict_swil()

    for k,v in ea_dict.items():
        seed, irm, name = parse_fname_swil(str(k))
        name = name[0]+'3'

        try:
            df = create_df_from_ea(v, "charts/episodic_return")    
            hparams = get_hparams_from_ea(v)

            n_demos = int(hparams['n_demos'])
            disc_nonlin = hparams['disc_nonlin']

            if 'sac' in str(k):
                df_dict['sac'][name][irm][n_demos][disc_nonlin][seed] = df 
                df_dict_max['sac'][name][irm][n_demos][disc_nonlin][seed] = max(df["Value"])
                hp_dict['sac'][name][irm][n_demos][disc_nonlin][seed] = hparams

            if 'ppo' in str(k):
                df_dict['ppo'][name][irm][n_demos][disc_nonlin][seed] = df 
                #print(k, max(df["Value"]))
                df_dict_max['ppo'][name][irm][n_demos][disc_nonlin][seed] = max(df["Value"])
                hp_dict['ppo'][name][irm][n_demos][disc_nonlin][seed] = hparams

            if 'sac' in str(k):
                df_dict_flat[f"sac_{name}_{irm}_seed{seed}"] = df 
            if 'ppo' in str(k):
                df_dict_flat[f"ppo_{name}_{irm}_seed{seed}"] = df
        except Exception as e: 
            print(e, k)
            pass
        
    return df_dict, df_dict_max, hp_dict

def get_dicts_from_ea(ea_dict, limit_length=-1):
    def parse_fname_il(fname):    
        intgr = re.findall(r'\d+', fname)
        floats = re.findall(r"\d+\.\d+", fname)
        name = re.findall(r"[a-zA-Z0-9]*-[-a-zA-Z0-9]", fname)

        m = re.search('seed_(\d+)', fname, re.IGNORECASE)
        seed = int(m.group(1))
        m = re.search('irm(\d+)', fname, re.IGNORECASE)
        irm = int(m.group(1))

        if len(floats) > 0:
            irm_coeff = floats[0]
        else:
            irm_coeff = 0
        
        return int(seed), float(irm_coeff), name

    df_dict = {}

    df_dict['sac'] = init_results_dict()
    df_dict['ppo'] = init_results_dict()

    df_dict_flat = {}
    df_dict_max = {}
    irm_coeffs = [0.0, 0.01, 0.1, 1.0, 10.0, 50.0, 100.0]

    df_dict_max['sac'] = init_results_dict()
    df_dict_max['ppo'] = init_results_dict()

    hp_dict = {}
    hp_dict['sac'] = init_results_dict()
    hp_dict['ppo'] = init_results_dict()

    for k,v in ea_dict.items():
        seed, irm, name = parse_fname_il(str(k))
        name = name[0]+'3'
        df = create_df_from_ea(v, "charts/episodic_return")
        hparams = get_hparams_from_ea(v)

        n_demos = int(hparams['n_demos'])
        disc_nonlin = hparams['disc_nonlin']

        if 'sac' in str(k):
            df_dict['sac'][name][irm][n_demos][disc_nonlin][seed] = df 
            df_dict_max['sac'][name][irm][n_demos][disc_nonlin][seed] = max(df["Value"][:limit_length])
            hp_dict['sac'][name][irm][n_demos][disc_nonlin][seed] = hparams

        if 'ppo' in str(k):
            df_dict['ppo'][name][irm][n_demos][disc_nonlin][seed] = df 
            #print(k, max(df["Value"]))
            df_dict_max['ppo'][name][irm][n_demos][disc_nonlin][seed] = max(df["Value"][:limit_length])
            hp_dict['ppo'][name][irm][n_demos][disc_nonlin][seed] = hparams

        if 'sac' in str(k):
            df_dict_flat[f"sac_{name}_{irm}_seed{seed}"] = df 
        if 'ppo' in str(k):
            df_dict_flat[f"ppo_{name}_{irm}_seed{seed}"] = df

    # add mean and std columns:
    for k,v in ea_dict.items():
        seed, irm, name = parse_fname(str(k))
        name = name[0]+'3'

        if 'sac' in str(k):
            df_dict_max['sac'][name][irm][n_demos][disc_nonlin]['mean']= np.mean(list(df_dict_max['sac'][name][irm][n_demos][disc_nonlin].values()))
            df_dict_max['sac'][name][irm][n_demos][disc_nonlin]['std'] = np.std(list(df_dict_max['sac'][name][irm][n_demos][disc_nonlin].values()))
        if 'ppo' in str(k):
            df_dict_max['ppo'][name][irm][n_demos][disc_nonlin]['mean'] = np.mean(list(df_dict_max['ppo'][name][irm][n_demos][disc_nonlin].values()))
            df_dict_max['ppo'][name][irm][n_demos][disc_nonlin]['std'] = np.std(list(df_dict_max['ppo'][name][irm][n_demos][disc_nonlin].values()))
        
    #for algo in ['sac', 'ppo']
    #    for task in ['Ant-v3', 'Hopper-v3', 'HalfCheetah-v3', 'Walker2d-v3']:
    return df_dict, df_dict_max, hp_dict

In [ ]:
df_dict_swil = get_dicts_from_ea_swil(ea_dict_swil)
df_dict_airl = get_dicts_from_ea(ea_dict_airl, limit_length=1000)
df_dict_gail = get_dicts_from_ea(ea_dict_gail, limit_length=1000)

In [ ]:
#print(json.dumps(df_dict_gail[1]['ppo']['Hopper-v3'][0.0][10]["tanh"], indent=4))
#print(json.dumps(df_dict_swil[1]['ppo']['Hopper-v3'], indent=4))
import json
print(json.dumps(df_dict_swil[1]['ppo']['Walker2d-v3'], indent=4))

In [ ]:
X = ['Group A','Group B','Group C','Group D']
Ygirls = [10,20,20,40]
Zboys = [20,30,25,30]
  
X_axis = np.arange(len(X))
  
plt.bar(X_axis - 0.2, Ygirls, 0.4, label = 'Girls')
plt.bar(X_axis + 0.2, Zboys, 0.4, label = 'Boys')
  
plt.xticks(X_axis, X)
plt.xlabel("Groups")
plt.ylabel("Number of Students")
plt.title("Number of Students in each group")
plt.legend()
plt.show()

In [ ]:
df_dict_airl[1]['ppo']['Hopper-v3'][50][25]['prelu']

In [ ]:
df_dict_swil[1]['ppo']['Hopper-v3'][50][25]['prelu']

In [ ]:
df_dict_swil[1]['ppo']['Hopper-v3'][50][25]['relu']

In [ ]:
fig, axs = plt.subplots(1,3, figsize=(20,4))
nonlin="prelu"
plt.style.use('ggplot')

for i, env in enumerate(['HalfCheetah-v3', 'Hopper-v3', 'Walker2d-v3']):
    X = ['gail','airl','swil:np1','swil:np5','swil:np20','swil:np50']
    X_axis = np.arange(len(X))

    axs[i].set_title(env)
    #plt.gca()
    #plt.xticks(X_axis, X)
    print(X)
    axs[i].set_xticks(X_axis,X)

    for j,nd in enumerate([4,10,25]):
        Y = []
        Y_ = []
        gail_val = list(df_dict_gail[1]['ppo'][env][0.0][nd][nonlin].values())
        airl_val = list(df_dict_airl[1]['ppo'][env][0.0][nd][nonlin].values())
        if len(gail_val) > 0:
            Y.append(np.mean(gail_val))
            Y_.append(np.std(gail_val))
        else:
            Y.append(0)
            Y_.append(0)
        
        if len(airl_val) > 0:
            Y.append(np.mean(airl_val))
            Y_.append(np.std(airl_val))
        else:
            Y.append(0)
            Y_.append(0)

        n_proj = [1,5,20,50]
        for n_p in n_proj:
            swil_val = list(df_dict_swil[1]['ppo'][env][n_p][nd][nonlin].values())
            if len(swil_val) > 0:
                Y.append(np.mean(swil_val))
                Y_.append(np.std(swil_val))
            else:
                Y.append(0)            
                Y_.append(0)
                
        print(Y, Y_)
                
        axs[i].bar(X_axis-0.2+j*0.2, Y, 0.2, label=f"{nd} trajectories") 
        axs[i].errorbar(X_axis-0.2+j*0.2, Y, yerr=Y_, fmt=".", color='k') 

        #plt.gca()
        #plt.xticks(X_axis, X)

axs[0].set_ylabel("Cumulative reward")

plt.legend()
plt.tight_layout()
plt.savefig(f"il_bar_ae_{nonlin}.png")

In [ ]:
plt.plot(df_dict_swil[0]['ppo']['HalfCheetah-v3'][50][4]["relu"][515].Value)

In [ ]:
# TODO: filter length!!

In [ ]:
plt.plot(df_dict_gail[0]['ppo']['HalfCheetah-v3'][0.0][4]["relu"][234].Value[:1000])

In [ ]:
hp_dict['ppo']['Hopper-v3'][0.0][322]['disc_nonlin']

In [ ]:
for d_nl in ['tanh', 'relu', 'prelu']:
    fig, axs = plt.subplots(4,3)
    for i, env in enumerate(df_dict['ppo'].keys()):
        for j, n_d in [4,10,25]:
            df_dict['ppo'][env][irm][n_demos][d_nl][seed]
            axs[i,j].set_title(f"{env}_nd:{n_d}")
            axs[i,j].plot(

        

In [ ]:
for k,v in df_dict['ppo']['Walker2d-v3'][50.0][10]['prelu'].items():
    print(k, v)
    
    


In [ ]:
res_gail_df = pd.DataFrame.from_dict({(i,j, k): df_dict_max[i][j][k] 
                           for i in df_dict_max.keys() 
                           for j in df_dict_max[i].keys()
                           for k in df_dict_max[i][j].keys()},
                       orient='index')

In [ ]:
res_gail_df.to_pickle("results_gail_df.pkl")

## GAIL

In [ ]:
pd.read_pickle('runs/results_gail_df.pkl')

## AIRL

In [ ]:
df_airl = pd.read_pickle('runs/results_airl_df.pkl')

In [ ]:
df_airl.keys()

In [ ]:
df_airl_flat = df_airl.reset_index()

In [ ]:
df_airl['std'] = np.std(df_airl[[234, 72, 894, 515, 322]].to_numpy(),axis=-1)

In [ ]:
df_airl

In [ ]:
df_airl_flat.columns = ['algo', 'env', 'irm_c', 234, 72, 894, 515, 322]

In [ ]:
tags = ["charts/episodic_return", "irl/gail_bce_loss"]
run_dir = Path("runs/airl_folders")      
#df_dict = load_dirs(run_dir, tags)

In [ ]:
dlist = []

for d in run_dir.iterdir():
    if 'max.pkl' in str(d):
        dlist.append(pkl.load(open(d,'rb')))

In [ ]:
from functools import reduce

def merge(a, b, path=None):
    "merges b into a"
    if path is None: path = []
    for key in b:
        if key in a:
            if isinstance(a[key], dict) and isinstance(b[key], dict):
                merge(a[key], b[key], path + [str(key)])
            elif a[key] == b[key]:
                pass # same leaf value
            else:
                raise Exception('Conflict at %s' % '.'.join(path + [str(key)]))
        else:
            a[key] = b[key]
    return a

def combine_into(d: dict, combined: dict) -> None:
    for k, v in d.items():
        if isinstance(v, dict):
            combine_into(v, combined.setdefault(k, {}))
        else:
            combined[k] = v



In [ ]:
d_all = {}
for d in dlist:
    combine_into(d, d_all)


In [ ]:
res_airl_df =pd.DataFrame.from_dict({(i,j, k): d_all[i][j][k] 
                           for i in d_all.keys() 
                           for j in d_all[i].keys()
                           for k in d_all[i][j].keys()},
                       orient='index')

In [ ]:
res_airl_df

In [ ]:
res_airl_df.to_pickle("results_airl_df.pkl")

In [ ]:
dflist = []
for d in dlist:
    df = pd.DataFrame.from_dict({(i,j, k): d[i][j][k] 
                           for i in d.keys() 
                           for j in d[i].keys()
                           for k in d[i][j].keys()},
                       orient='index')
    dflist.append(df)

In [ ]:
d = {}
d['sac'] = {'Ant-v3': {0.0:{},0.01:{},0.1:{},1.0:{},10.0:{},50.0:{},100.0:{}},
                  'HalfCheetah-v3': {0.0:{},0.01:{},0.1:{},1.0:{},10.0:{},50.0:{},100.0:{}},
                  'Hopper-v3': {0.0:{},0.01:{},0.1:{},1.0:{},10.0:{},50.0:{},100.0:{}},
                  'Walker2d-v3': {0.0:{},0.01:{},0.1:{},1.0:{},10.0:{},50.0:{},100.0:{}}}
d['ppo'] = {'Ant-v3': {0.0:{},0.01:{},0.1:{},1.0:{},10.0:{},50.0:{},100.0:{}},
                  'HalfCheetah-v3': {0.0:{},0.01:{},0.1:{},1.0:{},10.0:{},50.0:{},100.0:{}},
                  'Hopper-v3': {0.0:{},0.01:{},0.1:{},1.0:{},10.0:{},50.0:{},100.0:{}},
                  'Walker2d-v3': {0.0:{},0.01:{},0.1:{},1.0:{},10.0:{},50.0:{},100.0:{}}}

df_all = pd.DataFrame.from_dict({(i,j, k): d[i][j][k] 
                           for i in d.keys() 
                           for j in d[i].keys()
                           for k in d[i][j].keys()},
                       orient='index')
for df in dflist:
    df_all.join(df)